# OTTO · Retrieval quality meets search cost

**Experiment 06 / Fold 0 ANN benchmark / Measured and independently audited**

Approximate search retained 99.16–99.42% of exact top-800 neighbors on the reserved confirmation sessions. On the same tuning queries, its median CPU search was 5.35–5.63 times faster. The full-fold official Recall@20 decreased by 0.118 percentage points. These are measured trade-offs from a completed experiment, not a claim of state-of-the-art performance.

The model, catalogue embeddings, and exact predictions were frozen. All 96 full-fold ANN prediction parts are durable. The completed, independently audited baseline comparison adds **1.027 percentage points** of candidate ceiling at K=800, versus 1.029 points for exact search. This is candidate coverage before learned ranking, not a final recommendation score.

**Evaluation scope:** Fold 0 already selected the model checkpoint; these results are exploratory validation. Search timings exclude query encoding, network, and loading. Confirmation queries did not select `nprobe`.

Use the pinned analysis kernel in `notebooks/requirements.txt`.

[Operating guide](../docs/ANN_BENCHMARK.md) · [Raw report](../reports/metrics/two_tower_fold0_ann.json) · [Independent audit](../reports/metrics/two_tower_fold0_ann_audit.json)

This notebook pins the published, audited run. Local launcher pointers do not silently replace its evidence.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import time
import tomllib

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import HTML, Image, display

STARTED = time.perf_counter()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "configs/two_tower_ann.toml").is_file())
configuration = tomllib.loads((ROOT / "configs/two_tower_ann.toml").read_text())["benchmark"]
OBJECTIVES = ("clicks", "carts", "orders")
COLORS = {"clicks": "#3366b0", "carts": "#17806e", "orders": "#c05b43"}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.facecolor": "#f7f9fc", "axes.facecolor": "#f7f9fc"})

report_path = ROOT / "reports/metrics/two_tower_fold0_ann.json"
report = json.loads(report_path.read_text())
audit_path = ROOT / "reports/metrics/two_tower_fold0_ann_audit.json"
audit = json.loads(audit_path.read_text())
receipt = json.loads(Path(str(report_path) + ".json").read_text())
assert report["status"] == audit["status"] == "passed"
assert report["input_id"] == receipt["input_id"] == audit["input_id"]
assert hashlib.sha256(report_path.read_bytes()).hexdigest() == receipt["sha256"] == audit["metrics_sha256"]
configuration = report["contract"]["settings"]
print(f"[{datetime.now(timezone.utc).isoformat()}] notebook_start")
print(f"run_id={report['input_id']}")
print(f"Independent ANN audit: {audit['verified_count_parts']} parts, {audit['sessions']:,} sessions")
comparison_path = ROOT / "reports/metrics/two_tower_fold0_ann_comparison.json"
comparison = json.loads(comparison_path.read_text())
comparison_audit = json.loads((ROOT / "reports/metrics/two_tower_fold0_ann_comparison_audit.json").read_text())
assert comparison["status"] == comparison_audit["status"] == "passed"
assert comparison["input_id"] == comparison_audit["input_id"]
assert comparison["prediction_input_id"] == report["input_id"]
assert hashlib.sha256(comparison_path.read_bytes()).hexdigest() == comparison_audit["metrics_sha256"]
print(f"Independent baseline-comparison audit: {comparison_audit['verified_parts']} parts, "
      f"{comparison_audit['bootstrap_iterations_verified']} paired draws")


## Execution and recovery evidence

The accepted benchmark completed with all 96 full-fold prediction parts. Its CPU baseline comparison then completed in 400.735 seconds including control work, with all 32 count parts in S3. A later benchmark repeated identical worker bytes after a reporting commit; that completed attempt and its 3,040 billable seconds remain in the history.

The launcher now compares the complete saved experiment contract except the enclosing Git commit. Matching worker bytes, inputs, and settings reuse the original run and checkpoint namespace. Changed experimental inputs still require a new run. Failed or stopped work retries only with an explicit start.


In [ ]:
launch_path = ROOT / "reports/metrics/two_tower_fold0_ann_launch.json"
if launch_path.is_file():
    launch = json.loads(launch_path.read_text())
    attempts = [*launch.get("previous_attempts", []), launch, *launch.get("additional_completed_attempts", [])]
    display(pd.DataFrame([{
        "Attempt": i + 1, "Status": row["status"], "Phase": row["phase"],
        "AWS billable seconds": row["billable_seconds"],
        "Committed reference parts": row.get("committed_reference_parts", 0),
    } for i, row in enumerate(attempts)]).style.hide(axis="index"))
    print(launch["cause"])
    print(launch["interpretation"])
catalogue = json.loads((ROOT / "reports/metrics/two_tower_fold0_catalogue.json").read_text())
display(pd.DataFrame([
    ("Actual catalogue items validated", f"{catalogue['catalogue_items']:,}"),
    ("IDs sorted", str(catalogue["ids_sorted"])),
    ("Embedding row order", catalogue["row_order"]),
    ("Catalogue check", catalogue["status"]),
    ("Reference count parts verified", catalogue["production_reference_reuse_check"]["parts_validated"]),
], columns=["Input contract", "Observed value"]).style.hide(axis="index"))

recovery = launch.get("recovery_validation", {})
if recovery:
    production = recovery["production_s3_restore"]
    display(pd.DataFrame([
        ("Root tests passed", recovery["root_tests_passed"]),
        ("Worker tests passed", recovery["worker_tests_passed"]),
        ("Worker Python warning policy", recovery["worker_python_warnings"]),
        ("Production S3 restore", production["status"]),
        ("Recomputed saved reference part", production["recomputed"]),
        ("AWS container loader warning", launch["container_warning"]["status"]),
    ], columns=["Recovery check", "Observed value"]).style.hide(axis="index"))

if report is not None:
    display(pd.DataFrame([
        ("Reference parts restored", launch["restored_reference_parts"]),
        ("Full-fold prediction parts committed", launch["committed_prediction_parts"]),
        ("AWS billable seconds", launch["billable_seconds"]),
        ("Checkpoint objects", launch["ann_checkpoint_objects"]),
        ("Checkpoint storage GiB", f"{launch['ann_checkpoint_bytes']/1024**3:.2f}"),
        ("Independent count audit", audit["status"] if audit else "Pending for this run"),
    ], columns=["Completed-run evidence", "Value"]).style.hide(axis="index"))


## Evidence at a glance

Official Recall@20 evaluates the ordered recommendation list. Candidate ceilings describe the best a later ranker could recover from a larger pool. They answer different questions.

In [ ]:
previous = json.loads((ROOT / "reports/metrics/two_tower_fold0_retrieval.json").read_text())
exact20 = next(row for row in previous["points"] if row["neural_k"] == 20)
union800 = next(row for row in previous["points"] if row["neural_k"] == 800)
selected = report["tuning"][str(report["selected_nprobe"])]
speedups = [report["exact_cpu_latency_on_tuning_queries"][o]["p50_ms"] /
            selected["search"][o]["latency"]["p50_ms"] for o in OBJECTIVES]
delta = report["full_ann_ranking"]["weighted_recall_at_20"] - report["full_reference_ranking"]["weighted_recall_at_20"]
cards = [
    ("ANN · OFFICIAL RECALL@20", f"{report['full_ann_ranking']['weighted_recall_at_20']:.3%}",
     f"{delta*100:+.3f} percentage points versus exact neural search"),
    ("MEDIAN CPU SEARCH SPEEDUP", f"{min(speedups):.2f}–{max(speedups):.2f}×",
     "Same tuning queries; search + reranking only"),
    ("CONFIRMATION · TOP-800 OVERLAP",
     f"{min(report['confirmation']['search'][o]['fidelity']['800'] for o in OBJECTIVES):.2%}+",
     "All objectives passed the prospective 98% target"),
]
display(HTML('<div style="display:flex;flex-wrap:wrap;gap:14px">' + ''.join(
    f'<div style="flex:1;min-width:240px;background:#f7f9fc;border:1px solid #dce3ec;'
    f'border-radius:12px;padding:20px"><div style="color:#536174;font-size:11px;'
    f'font-weight:700">{title}</div><div style="color:#17324f;font-size:26px;'
    f'font-weight:700;margin:12px 0">{value}</div><div style="color:#536174;'
    f'font-size:12px">{note}</div></div>' for title, value, note in cards) + '</div>'))
ann800 = next(row for row in comparison["points"] if row["neural_k"] == 800)
print(f"ANN addition: +{100*ann800['weighted_incremental_ceiling']:.3f} pp candidate ceiling at K=800. "
      f"Exact addition: +{100*union800['weighted_incremental_ceiling']:.3f} pp. "
      "The ANN baseline comparison is complete. See notebook 08 for the measured compressed-pool ranker baseline; certified neural-source ranking remains unmeasured.")


## Freeze the decision before looking at the result

The smallest `nprobe` meeting the **prospective 98% mean top-800 overlap target for each objective** is selected using tuning queries only. That one setting is evaluated on the disjoint confirmation queries. A failed target is an informative experiment; no extra folds or automatic retries are launched. Full-fold ANN export runs only after confirmation passes.

Fold 0 already selected the model checkpoint. The reserved ANN confirmation split does not make this an untouched model test. Full-fold metrics include the ANN tuning sessions.

In [ ]:
display(pd.DataFrame([
    ("Tuning / confirmation sessions", f"{configuration['sample_sessions']//2:,} / {configuration['sample_sessions']//2:,}"),
    ("Index", f"IVFFlat · {configuration['nlist']:,} centroids · original FP32 vectors"),
    ("Training vectors / iterations", f"{configuration['train_items']:,} / {configuration['train_iterations']}"),
    ("Probe sweep", str(configuration["probes"])),
    ("Per-objective target", f"{configuration['target_overlap']:.0%} mean top-800 overlap"),
    ("Search CPU threads / batch size", f"{configuration['threads']} / {configuration['batch_size']}"),
    ("Latency sample", f"{configuration['latency_queries']} queries × {configuration['latency_repeats']} repeats; {configuration['warmup_queries']} warm-up calls"),
    ("Full-fold export", "Enabled after confirmation passes"),
], columns=["Frozen run contract", "Setting"]).style.hide(axis="index"))


## Primary model quality: the official project metric

For each objective, sum top-20 positive hits over sessions and divide by the sum of `min(20, true-item count)`. Combine objectives using the [official OTTO weights](https://github.com/otto-de/recsys-dataset/blob/main/KAGGLE.md): **clicks 0.10, carts 0.30, orders 0.60**. Unknown catalogue positives remain misses.

NDCG@20, MRR@20, hit rate, and precision are diagnostics averaged over labeled sessions separately for each objective. They are not replacements for the official metric. The bootstrap resamples whole sessions jointly across objectives; its interval does not account for model selection.

In [ ]:
if report is None:
    display(pd.DataFrame([
        {"Objective": o.title(), "Exact neural Recall@20": exact20["objectives"][o]["neural_ceiling"],
         "ANN Recall@20": "Pending"} for o in OBJECTIVES
    ]).style.format({"Exact neural Recall@20": "{:.3%}"}).hide(axis="index"))
    print("NDCG@20, MRR@20 and ANN differences will be read from the managed report.")
else:
    cohorts = [("Full-fold exact", report["full_reference_ranking"])]
    if report.get("full_ann_ranking"):
        cohorts.append(("Full-fold ANN", report["full_ann_ranking"]))
    if report.get("confirmation"):
        cohorts.extend([
            ("Confirmation exact", report["confirmation"]["exact_ranking"]),
            ("Confirmation ANN", report["confirmation"]["ranking"]),
        ])
    display(pd.DataFrame([
        {"Cohort / method": name, "Sessions": data["sessions"],
         "Official weighted Recall@20": data["weighted_recall_at_20"]}
        for name, data in cohorts
    ]).style.format({"Official weighted Recall@20": "{:.3%}", "Sessions": "{:,}"}).hide(axis="index"))
    diagnostics = []
    for name, data in cohorts:
        for objective, values in data["objectives"].items():
            diagnostics.append({"Cohort / method": name, "Objective": objective,
                **{key: values[key] for key in ("recall_at_20", "ndcg_at_20", "mrr_at_20",
                                               "hit_rate_at_20", "precision_at_20")}})
    display(pd.DataFrame(diagnostics).style.format({key: "{:.4f}" for key in diagnostics[0]
            if key not in {"Cohort / method", "Objective"}}).hide(axis="index"))
    if report.get("full_ann_paired_uncertainty"):
        lo, hi = report["full_ann_paired_uncertainty"]["weighted_recall_at_20_delta_ci95"]
        delta = report["full_ann_ranking"]["weighted_recall_at_20"] - report["full_reference_ranking"]["weighted_recall_at_20"]
        print(f"Full-fold ANN − exact: {delta*100:+.3f} pp; paired 95% interval [{lo*100:+.3f}, {hi*100:+.3f}] pp")

if report is not None:
    exact = report["full_reference_ranking"]
    ann = report["full_ann_ranking"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    x = np.arange(4)
    ev = [exact["objectives"][o]["recall_at_20"] for o in OBJECTIVES] + [exact["weighted_recall_at_20"]]
    av = [ann["objectives"][o]["recall_at_20"] for o in OBJECTIVES] + [ann["weighted_recall_at_20"]]
    axes[0].bar(x-.18, ev, .36, label="Exact", color="#7c8fa8")
    axes[0].bar(x+.18, av, .36, label="ANN · nprobe 256", color="#17806e")
    axes[0].set_xticks(x, ["Clicks", "Carts", "Orders", "Weighted"])
    axes[0].set_ylim(0, .40)
    axes[0].yaxis.set_major_formatter(PercentFormatter(1))
    axes[0].set_ylabel("Official Recall@20")
    axes[0].set_title("Measured ranking quality", loc="left", fontweight="bold")
    axes[0].legend(frameon=False, fontsize=9)
    x = np.arange(3)
    exact_p95 = [report["exact_cpu_latency_on_tuning_queries"][o]["p95_ms"] for o in OBJECTIVES]
    ann_p95 = [selected["search"][o]["latency"]["p95_ms"] for o in OBJECTIVES]
    bars = axes[1].bar(x-.18, exact_p95, .36, color="#7c8fa8")
    axes[1].bar_label(bars, fmt="%.1f", padding=3, fontsize=9)
    bars = axes[1].bar(x+.18, ann_p95, .36, color="#17806e")
    axes[1].bar_label(bars, fmt="%.1f", padding=3, fontsize=9)
    axes[1].set_xticks(x, [o.title() for o in OBJECTIVES])
    axes[1].set_ylim(0, 90)
    axes[1].set_ylabel("Warm batch-1 p95 search (ms)")
    axes[1].set_title("Faster search on matched tuning queries", loc="left", fontweight="bold")
    for ax in axes:
        ax.grid(axis="y", alpha=.2)
        ax.set_axisbelow(True)
    lo, hi = report["full_ann_paired_uncertainty"]["weighted_recall_at_20_delta_ci95"]
    fig.suptitle("OTTO | Quality and search cost", x=.06, ha="left", fontsize=18,
                 fontweight="bold", color="#17324f")
    fig.text(.06, .025, f"103,468 sessions · ANN − exact: {delta*100:+.3f} pp "
             f"(paired 95% interval {lo*100:+.3f} to {hi*100:+.3f} pp)\n"
             "Latency excludes encoding, network, and loading. Fold 0 remains exploratory validation.",
             fontsize=9, color="#536174")
    fig.subplots_adjust(left=.07, right=.98, top=.80, bottom=.23, wspace=.30)
    fig.savefig(ROOT / "reports/figures/two_tower_ann_quality.png", dpi=180)
    display(Image(filename=str(ROOT / "reports/figures/two_tower_ann_quality.png")))
    plt.close(fig)


## Search frontier: fidelity versus latency

Each point is one **tuning** configuration. Latency is warm, batch-1 CPU search plus FP32 reranking of precomputed queries. It excludes the session encoder, network, and index loading. The chart must not be described as end-to-end serving latency.

The exact CPU reference uses Flat inner-product search on the same tuning queries and the same thread count. The confirmation result appears separately below; it never chooses the configuration. FAISS explains the [IVF probe trade-off](https://github.com/facebookresearch/faiss/wiki/Faster-search).

In [ ]:
if report is None:
    print("Search frontier pending. No latency or ANN overlap values have been measured for the full catalogue.")
else:
    frontier = []
    for probe, result in report["tuning"].items():
        for objective in OBJECTIVES:
            search = result["search"][objective]
            frontier.append({"nprobe": int(probe), "objective": objective,
                "overlap": search["fidelity"][str(report["contract"]["settings"]["candidate_depth"])],
                "p95_ms": search["latency"]["p95_ms"],
                "queries_per_second": search["batch_throughput_queries_per_second"]})
    frontier = pd.DataFrame(frontier)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
    for ax, objective in zip(axes, OBJECTIVES):
        rows = frontier[frontier.objective == objective].sort_values("nprobe")
        ax.plot(rows.p95_ms, rows.overlap, "o-", color=COLORS[objective], linewidth=2)
        for row in rows.itertuples():
            ax.annotate(str(row.nprobe), (row.p95_ms, row.overlap),
                        xytext=(5, 6), textcoords="offset points", fontsize=9)
        ax.axhline(report["contract"]["settings"]["target_overlap"], color="#758397",
                   linestyle="--", linewidth=1)
        ax.set_title(objective.title(), loc="left", fontweight="bold")
        ax.set_xlabel("Warm batch-1 p95 search (ms)")
        ax.grid(alpha=0.2)
        ax.yaxis.set_major_formatter(PercentFormatter(1))
    axes[0].set_ylabel("Mean exact top-K ID overlap")
    fig.suptitle("OTTO | ANN tuning frontier", x=0.06, ha="left", fontweight="bold", fontsize=18, color="#17324f")
    fig.text(0.06, 0.015, "Point labels = nprobe · Dashed line = prospective fidelity target\nCPU search + reranking only; encoding and network are excluded.", fontsize=9, color="#536174")
    fig.subplots_adjust(left=0.07, right=0.98, top=0.80, bottom=0.24, wspace=0.22)
    figure_path = ROOT / "reports/figures/two_tower_ann.png"
    figure_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_path, dpi=180, facecolor=fig.get_facecolor())
    display(Image(filename=str(figure_path)))
    plt.close(fig)
    display(frontier.style.format({"overlap": "{:.2%}", "p95_ms": "{:.3f}",
                                   "queries_per_second": "{:.1f}"}).hide(axis="index"))
    display(pd.DataFrame([
        {"Objective": o, "Exact CPU p50 ms": row["p50_ms"], "Exact CPU p95 ms": row["p95_ms"],
         "Exact CPU p99 ms": row["p99_ms"], "Timing observations": row["samples"]}
        for o, row in report["exact_cpu_latency_on_tuning_queries"].items()
    ]).style.format(precision=3).hide(axis="index"))


## Confirmation and engineering cost

A retained index or prediction part is reused only after its checksum and run identity match. Data is uploaded before its receipt. Missing/corrupt parts are recomputed; valid completed work survives across managed workers.

Latency observations can come from a previous compatible attempt. Retained compute time is the sum of committed part timings, not total paid time. AWS per-execution billable seconds live in the durable `control/executions/` records and are printed by the monitor.

In [ ]:
if report is None:
    print("Confirmation, index sizes, memory use, and measured build/search times are pending.")
else:
    print(f"selected_nprobe={report['selected_nprobe']} confirmation_fidelity_passed={report['confirmation_fidelity_passed']}")
    if report.get("confirmation"):
        rows = []
        for objective, row in report["confirmation"]["search"].items():
            rows.append({"Objective": objective, "Overlap@20": row["fidelity"]["20"],
                "Overlap@K": row["fidelity"][str(report["contract"]["settings"]["candidate_depth"])],
                "p50 ms": row["latency"]["p50_ms"], "p95 ms": row["latency"]["p95_ms"],
                "p99 ms": row["latency"]["p99_ms"]})
        display(pd.DataFrame(rows).style.format({"Overlap@20": "{:.2%}", "Overlap@K": "{:.2%}",
            "p50 ms": "{:.3f}", "p95 ms": "{:.3f}", "p99 ms": "{:.3f}"}).hide(axis="index"))
    display(pd.DataFrame([
        {"Objective": o, "Index GiB": row["index_bytes"]/1024**3,
         "Retained build seconds": row["retained_build_compute_seconds"],
         "Load seconds this attempt": row["load_seconds_this_attempt"], "Index shards": row["shards"]}
        for o, row in report["index_builds"].items()
    ]).style.format(precision=3).hide(axis="index"))
    display(pd.DataFrame([
        ("Worker", report["contract"]["instance_type"]),
        ("Encoder device", report["contract"].get("encoder_device", "Unrecorded")),
        ("Search CPU threads", report["contract"]["settings"]["threads"]),
        ("Recorded Python / PyTorch", f"{report['contract']['python']} / {report['contract']['torch']}"),
        ("Recorded CUDA / FAISS", f"{report['contract']['cuda_runtime']} / {report['contract']['faiss']}"),
        ("AWS billable seconds", launch["billable_seconds"]),
        ("Peak process RSS MiB", report["peak_rss_mib"]),
        ("Current attempt seconds", report["elapsed_seconds_this_attempt"]),
        ("Retained artifact compute seconds", report["retained_artifact_compute_seconds"]),
        ("Full-fold prediction export", report.get("prediction_export")),
    ], columns=["Resource / evidence", "Value"]).style.hide(axis="index"))


## Complementary candidate value: measured

The fixed baseline contains revisit, co-visitation, and Item2Vec candidates. Adding the first 800 ANN candidates raises its weighted ideal top-20 ceiling from **73.154% to 74.181%**: **+1.027 percentage points**, with a 95% paired session interval of **+0.928 to +1.117 points**. The same exact-search addition was +1.029 points.

The curves below are candidate ceilings, not ranked Recall@20. Each interval measures addition to the baseline; overlapping intervals do not establish ANN/exact equivalence. The frozen baseline hashes and session cohort agree. This comparison cannot identify whether the exact same exclusive positives were retained from aggregate counts alone.


In [ ]:
assert comparison["contract"]["baseline_checksums"] == previous["contract"]["baseline_checksums"]
assert comparison["sessions"] == previous["sessions"]
rows = []
for label, result in (("Exact", previous), ("ANN", comparison)):
    for point in result["points"]:
        low, high = point["weighted_incremental_ci95"]
        rows.append({"Search": label, "Neural K": point["neural_k"],
                     "Base ceiling": point["weighted_base_ceiling"],
                     "Union ceiling": point["weighted_union_ceiling"],
                     "Gain (pp)": 100*point["weighted_incremental_ceiling"],
                     "95% low (pp)": 100*low, "95% high (pp)": 100*high})
frontier_comparison = pd.DataFrame(rows)
display(frontier_comparison.style.format({"Base ceiling":"{:.3%}", "Union ceiling":"{:.3%}",
    "Gain (pp)":"{:.3f}", "95% low (pp)":"{:.3f}", "95% high (pp)":"{:.3f}"}).hide(axis="index"))
fig, axes = plt.subplots(1, 2, figsize=(13.4, 5.3))
for label, color, style in (("Exact", "#3366b0", "--"), ("ANN", "#17806e", "-")):
    rows = frontier_comparison[frontier_comparison["Search"] == label]
    axes[0].plot(rows["Neural K"], rows["Gain (pp)"], style, color=color,
                 marker="o", markersize=5, label=label, linewidth=2)
    axes[0].fill_between(rows["Neural K"], rows["95% low (pp)"], rows["95% high (pp)"], color=color, alpha=.10)
axes[0].set(xlabel="Neural candidates added per objective", ylabel="Weighted candidate-ceiling gain (pp)",
            title="Additional coverage across candidate budgets")
axes[0].legend(frameon=False)
x = np.arange(3)
for offset, label, result, color in ((-.18, "Exact", union800, "#3366b0"), (.18, "ANN", ann800, "#17806e")):
    values = np.array([100*result["objectives"][o]["incremental_ceiling"] for o in OBJECTIVES])
    intervals = np.array([result["objectives"][o]["incremental_ci95"] for o in OBJECTIVES])*100
    axes[1].bar(x+offset, values, width=.34, color=color, label=label,
                yerr=np.vstack([values-intervals[:,0], intervals[:,1]-values]), capsize=4,
                error_kw={"elinewidth":1, "ecolor":"#26384a"})
axes[1].set(xticks=x, xticklabels=[o.title() for o in OBJECTIVES],
            ylabel="Candidate-ceiling gain (pp)", title="Added coverage by objective · K=800")
axes[1].legend(frameon=False)
for ax in axes:
    ax.grid(axis="y", alpha=.18)
    ax.set_axisbelow(True)
fig.suptitle("OTTO · ANN preserves nearly all measured incremental coverage", x=.055, ha="left", fontsize=17, fontweight="bold")
fig.text(.055, .025, "Fold 0 exploratory validation · 103,468 paired sessions · 500 bootstrap draws · Candidate ceilings, not final ranked Recall@20", fontsize=9, color="#536174")
fig.tight_layout(rect=(.02, .07, .99, .90))
figure_path = ROOT / "reports/figures/two_tower_ann_comparison.png"
fig.savefig(figure_path, dpi=160, facecolor=fig.get_facecolor())
display(Image(filename=str(figure_path)))
plt.close(fig)


## Decision and next modeling stage

The first 30-feature, 100-candidate baseline ranker is now measured in [notebook 08](08_ranking_evaluation.ipynb). This notebook's larger-pool ANN ceilings are not that ranker's achieved score or candidate ceiling.

Next, compare compression budgets, source families and broad candidate feature groups through training-only screening and inner validation. ANN is a prospective additional source, not yet a certified feature in the independently selected ranker. Fold 0 selected the neural checkpoint; source fitting and checkpoint selection must exclude the relevant evaluation labels before a valid with/without-neural ranking comparison.

Retain candidate-generation misses in the denominator and use official weighted Recall@20 as the primary metric. Objective-level recall, NDCG, MRR, candidate coverage, paired uncertainty, latency and memory are complementary diagnostics. An untouched temporal evaluation and validated full-test submission remain separate milestones. No leaderboard or state-of-the-art result is claimed here.


In [ ]:
runtime = {"python": platform.python_version(), "numpy": np.__version__,
           "pandas": pd.__version__}
print(json.dumps(runtime, sort_keys=True))
print(f"[{datetime.now(timezone.utc).isoformat()}] notebook_complete "
      f"elapsed_seconds={time.perf_counter()-STARTED:.3f}")
